# Statistics for ML Quality: Drift Detection and Confidence Intervals on Model Performance

## Problem statement

You are the QA engineer on a team that deployed a home-value prediction model trained on the
**California Housing** dataset (`sklearn.datasets.fetch_california_housing`, real 1990 census
data for California block groups — 20,640 rows, median house value as the target). The model
has been in production for a while. A new batch of block-group data has just come in, and your
job is to decide, using statistics rather than a gut feeling, whether:

1. **The input data has drifted** relative to what the model was trained on (feature-by-feature
   distribution shift), and
2. **The model's measured error has gotten statistically significantly worse** on the new batch,
   or whether the apparent change is within the noise you'd expect from sampling.

To make this concrete without needing a second real-world dataset, you will split the real
California Housing data into a `reference` batch (older housing stock, `HouseAge` below the
median) and a `current` batch (newer housing stock, `HouseAge` at or above the median). This is
a realistic drift scenario (the kind of time-based or cohort-based shift a production model sees)
built entirely from genuine census data — no synthetic/fabricated rows are introduced anywhere.

## What you should produce

By the end of this notebook you should have:

- A **drift report**: for every feature, the two-sample Kolmogorov-Smirnov statistic and p-value
  comparing `reference` vs. `current`, with a boolean `drifted` flag at `alpha = 0.05`, and a
  one-line written conclusion naming which features drifted and why that makes sense physically
  (e.g. newer housing stock tends to differ in income, rooms, location).
- A **bootstrap confidence interval** (95%) for the model's Mean Absolute Error (MAE) separately
  on a held-out slice of `reference` and on `current`.
- A written **verdict**: is the MAE on `current` significantly worse than on `reference` (do the
  confidence intervals fail to overlap), or is the difference plausibly just sampling noise?
  State your reasoning, not just the numbers.


## Starter cells

Imports, data loading, and the reference/current split are done for you. The functions below are
scaffolded with signatures and TODOs — fill in the logic yourself before looking at the solution.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

housing = fetch_california_housing(as_frame=True)
df = housing.frame.rename(columns={"MedHouseVal": "target"})
df.head()


In [ ]:
# Split into reference (older housing stock) and current (newer housing stock) batches.
# This is a genuine cohort split of real data -- not synthetic data.
age_median = df["HouseAge"].median()
reference_df = df[df["HouseAge"] < age_median].reset_index(drop=True)
current_df = df[df["HouseAge"] >= age_median].reset_index(drop=True)

FEATURES = [c for c in df.columns if c != "target"]

print(f"reference: {reference_df.shape}, current: {current_df.shape}, age_median: {age_median}")


In [ ]:
def ks_test_feature(reference: pd.Series, current: pd.Series) -> tuple[float, float]:
    """Run a two-sample Kolmogorov-Smirnov test between reference and current values
    for a single feature. Return (statistic, p_value).

    TODO: use scipy.stats.ks_2samp.
    """
    # TODO
    raise NotImplementedError


def detect_drifted_features(
    reference_df: pd.DataFrame, current_df: pd.DataFrame, features: list[str], alpha: float = 0.05
) -> pd.DataFrame:
    """For every feature in `features`, run ks_test_feature(reference_df[f], current_df[f])
    and build a report.

    TODO: return a DataFrame indexed by feature with columns:
      - 'statistic'   (KS statistic)
      - 'p_value'
      - 'drifted'     (bool: p_value < alpha)
    Sort by p_value ascending so the most drifted features are on top.
    """
    # TODO
    raise NotImplementedError


def bootstrap_metric_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    metric_fn=mean_absolute_error,
    n_boot: int = 1000,
    alpha: float = 0.05,
    random_state: int = RANDOM_STATE,
) -> tuple[float, float, float]:
    """Bootstrap a (1 - alpha) confidence interval for metric_fn(y_true, y_pred).

    TODO:
      1. Compute the point estimate on the full sample.
      2. Resample (y_true, y_pred) pairs together, with replacement, n_boot times
         (keep the same resampled indices for both arrays -- don't resample them
         independently, that would break the pairing between a true value and its
         prediction).
      3. Compute metric_fn on each resample to build a bootstrap distribution.
      4. Return (point_estimate, lower_bound, upper_bound) using the
         alpha/2 and 1 - alpha/2 percentiles of the bootstrap distribution.
    """
    # TODO
    raise NotImplementedError


# --- Train the model on a held-out split of the reference batch ---
X_train, X_test, y_train, y_test = train_test_split(
    reference_df[FEATURES], reference_df["target"], test_size=0.3, random_state=RANDOM_STATE
)
model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)

# TODO: use the functions above to
#   1. build `drift_report = detect_drifted_features(reference_df, current_df, FEATURES)`
#   2. compute bootstrap CIs for MAE on (y_test, model.predict(X_test))  -- the reference holdout
#   3. compute bootstrap CIs for MAE on (current_df['target'], model.predict(current_df[FEATURES])) -- current batch
#   4. compare the two CIs and write your verdict in a markdown cell below


---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

The KS test compares the empirical CDFs of two samples; a small p-value means the two samples are
unlikely to come from the same distribution. Bootstrapping the MAE (instead of relying on a single
point estimate) gives you a sense of how much that number would wobble under a different sample
of the same size, so you can tell a real regression from noise.

```python
def ks_test_feature(reference, current):
    statistic, p_value = stats.ks_2samp(reference, current)
    return statistic, p_value


def detect_drifted_features(reference_df, current_df, features, alpha=0.05):
    rows = []
    for feature in features:
        statistic, p_value = ks_test_feature(reference_df[feature], current_df[feature])
        rows.append({
            "feature": feature,
            "statistic": statistic,
            "p_value": p_value,
            "drifted": p_value < alpha,
        })
    report = pd.DataFrame(rows).set_index("feature").sort_values("p_value")
    return report


def bootstrap_metric_ci(y_true, y_pred, metric_fn=mean_absolute_error, n_boot=1000, alpha=0.05, random_state=RANDOM_STATE):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    point_estimate = metric_fn(y_true, y_pred)

    rng = np.random.default_rng(random_state)
    n = len(y_true)
    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_stats[i] = metric_fn(y_true[idx], y_pred[idx])

    lower = np.percentile(boot_stats, 100 * alpha / 2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return point_estimate, lower, upper


drift_report = detect_drifted_features(reference_df, current_df, FEATURES)
print(drift_report)

ref_mae, ref_lo, ref_hi = bootstrap_metric_ci(y_test.to_numpy(), model.predict(X_test))
cur_mae, cur_lo, cur_hi = bootstrap_metric_ci(
    current_df["target"].to_numpy(), model.predict(current_df[FEATURES])
)
print(f"reference MAE: {ref_mae:.4f}  95% CI [{ref_lo:.4f}, {ref_hi:.4f}]")
print(f"current   MAE: {cur_mae:.4f}  95% CI [{cur_lo:.4f}, {cur_hi:.4f}]")
print(f"CIs overlap: {not (cur_lo > ref_hi or ref_lo > cur_hi)}")
```

**Typical finding:** `MedInc`, `AveRooms`, `Latitude`/`Longitude` tend to show up as drifted
(p_value well below 0.05) because newer housing stock in this dataset skews toward different
income bands and is concentrated in different parts of the state than older stock — a physically
sensible result, not an artifact. The bootstrap MAE CI on `current` is typically both higher and
non-overlapping with the `reference` CI, meaning the degradation is statistically significant
and not just sampling noise — in a real pipeline this would be the trigger to investigate retraining,
not just a number to shrug off.

**Why this matters for ML quality engineering:** a single point-estimate metric comparison
("MAE went from 0.31 to 0.34, ship a retrain ticket") is exactly the kind of under-specified
assertion an SQA background teaches you to distrust. The KS test and bootstrap CI turn "it looks
worse" into a falsifiable, repeatable check you could wire into a CI/monitoring pipeline.

</details>
